In [ ]:
# Sequences and Recurrence
# Generated from the canonical HTML manuscript. Run this cell first.
# Source: https://github.com/Shakeri-Lab/dl-book/blob/c058d1f401fd0ead3ae59a2a8730f95489a2d9aa/chapters/part3/10-sequences-rnn.qmd

from importlib.metadata import PackageNotFoundError, version as package_version
import hashlib as _bootstrap_hashlib
import os as _bootstrap_os
from pathlib import Path as _BootstrapPath
import subprocess as _bootstrap_subprocess
import sys as _bootstrap_sys
import urllib.request as _bootstrap_urlrequest

_BOOK_REVISION = 'c058d1f401fd0ead3ae59a2a8730f95489a2d9aa'
_PINNED_REQUIREMENTS = [
    "torch==2.12.1",
    "torchvision==0.27.1",
    "numpy==2.5.1",
    "matplotlib==3.11.1"
]
_BOOK_ASSETS = [
    {
        "path": "code/dlbook/__init__.py",
        "sha256": "5f31ed4ff3aac6a557697078bfc7b9048811de3c1ccc0dc19890a6b1499ffb06"
    },
    {
        "path": "code/dlbook/evaluation.py",
        "sha256": "f89bf796ee2f4482d3ce8cccfb7b3859f2bbc56ac1bfdfff9cee058b1fd786ee"
    },
    {
        "path": "code/dlbook/training.py",
        "sha256": "7102ac8d5c5aa6b416ea3bf0acec403df658373ad88697294a807f03c48fdf58"
    },
    {
        "path": "data/book-corpus-ch1-9.txt",
        "sha256": "b0fc23a513e37e7bcd78da04a03877345f6ebc850b91dbae260656d9924fb299"
    },
    {
        "path": "experiments/rivanna/results/wikitext/large-6050.json",
        "sha256": "f0fd278355f1e01750cbc27fadeed0019a7660627835b609bc3795d1d9c3bd1d"
    },
    {
        "path": "experiments/rivanna/results/wikitext/large-6051.json",
        "sha256": "2e8ca456e681202ced6e396563baf294c4266e5316bed83b6e2ee2dfb288bd74"
    },
    {
        "path": "experiments/rivanna/results/wikitext/large-6052.json",
        "sha256": "a1790329bdce1551b68ca364b29b78bf1270a95a8af9fa78a22d8755b0ef8da2"
    },
    {
        "path": "experiments/rivanna/results/wikitext/medium-6050.json",
        "sha256": "afcf408322ca16692f8c07e1a56d14b03923779c53bc7a9324a52f0e906571bb"
    },
    {
        "path": "experiments/rivanna/results/wikitext/medium-6051.json",
        "sha256": "4a2cca5edc04a80631e2636756ded27233fe76bcd0b513feb1d2596999caaf8a"
    },
    {
        "path": "experiments/rivanna/results/wikitext/medium-6052.json",
        "sha256": "7ab730282a85b1030aca90b51acf58908ad0961b3a51e142332f18b21585cbbd"
    },
    {
        "path": "experiments/rivanna/results/wikitext/small-6050.json",
        "sha256": "0ebdf38a356f3bbc856485d4bef3677f0ccbe1f1abb6b34a4b5edefa8122e5ce"
    },
    {
        "path": "experiments/rivanna/results/wikitext/small-6051.json",
        "sha256": "02c67886e243c69ab59b89bd3603a63feb591d93369f907677ee89b62ee87db5"
    },
    {
        "path": "experiments/rivanna/results/wikitext/small-6052.json",
        "sha256": "4228a5f73b1a78cdd17f74a807d6ee5048dcca1aa22ad5b1a37a0364f0bbbaaa"
    }
]

def _installed_requirement(requirement: str) -> bool:
    name, expected = requirement.split('==', 1)
    try:
        return package_version(name) == expected
    except PackageNotFoundError:
        return False

_missing_requirements = [
    item for item in _PINNED_REQUIREMENTS if not _installed_requirement(item)
]
if _missing_requirements:
    _bootstrap_install = _bootstrap_subprocess.run(
        [_bootstrap_sys.executable, '-m', 'pip', 'install', '--quiet',
         *_missing_requirements],
        check=False, capture_output=True, text=True,
    )
    if _bootstrap_install.returncode != 0:
        raise RuntimeError(_bootstrap_install.stdout + _bootstrap_install.stderr)

_bootstrap_base = _BootstrapPath(
    _bootstrap_os.environ.get(
        'DLBOOK_NOTEBOOK_ROOT',
        '/content' if _BootstrapPath('/content').is_dir()
        else str(_BootstrapPath.home() / '.cache'),
    )
)
_BOOK_ROOT = _bootstrap_base / f'dl-book-{_BOOK_REVISION[:12]}'
_RAW_ROOT = 'https://raw.githubusercontent.com/Shakeri-Lab/dl-book/' + _BOOK_REVISION + '/'
for _record in _BOOK_ASSETS:
    _destination = _BOOK_ROOT / _record['path']
    _destination.parent.mkdir(parents=True, exist_ok=True)
    _valid = (
        _destination.is_file()
        and _bootstrap_hashlib.sha256(_destination.read_bytes()).hexdigest()
        == _record['sha256']
    )
    if not _valid:
        _temporary = _destination.with_suffix(_destination.suffix + '.part')
        _bootstrap_urlrequest.urlretrieve(_RAW_ROOT + _record['path'], _temporary)
        _digest = _bootstrap_hashlib.sha256(_temporary.read_bytes()).hexdigest()
        if _digest != _record['sha256']:
            _temporary.unlink(missing_ok=True)
            raise RuntimeError(f"Checksum mismatch for {_record['path']}")
        _temporary.replace(_destination)

(_BOOK_ROOT / 'chapters/part3').mkdir(parents=True, exist_ok=True)
_bootstrap_sys.path.insert(0, str(_BOOK_ROOT / 'code'))
_bootstrap_os.chdir(_BOOK_ROOT / 'chapters/part3')

# Hidden manuscript support required by later learner-visible cells.
# Plot-only harnesses are not exported.
import torch
from torch import nn

assert _BOOK_ROOT.is_dir()

**Plan**

1. Prepare the inputs and fixed settings for the example.
2. Implement the recurrence, written as the loop it is.
3. Report or visualize the measured result.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# [1]
torch.manual_seed(6050)
V, H, T, B = 8, 16, 12, 4                       # vocab, hidden, time, batch
rnn = nn.RNN(V, H, batch_first=True)
x = torch.randn(B, T, V)
out_ref, _ = rnn(x)

W_xh, W_hh = rnn.weight_ih_l0, rnn.weight_hh_l0
b = rnn.bias_ih_l0 + rnn.bias_hh_l0
h = torch.zeros(B, H)                            # h_0 = 0, the standard start
outs = []
# [2]
for t in range(T):                               # the loop IS the architecture
    h = torch.tanh(x[:, t] @ W_xh.T + h @ W_hh.T + b)
    outs.append(h)
manual = torch.stack(outs, 1)
# [3]
print(f"manual loop vs. nn.RNN: max |diff| = {(manual - out_ref).abs().max():.1e}")

**Plan**

1. Define the reusable `grad_at_first_input` helper.
2. Prepare the inputs and fixed settings for the example.
3. Gradient reaching the first input vs. lag.

In [ ]:
# [1]
def grad_at_first_input(rec: nn.Module, lags: list[int],
                        vocab: int = 4) -> list[float]:
    norms = []
    for T in lags:
        generator = torch.Generator().manual_seed(100 + T)
        x = torch.randn(8, T, vocab, generator=generator, requires_grad=True)
        o, _ = rec(x)
        o[:, -1].sum().backward()
        norms.append(x.grad[:, 0].norm().item())    # blame on input #1
    return norms

# [2]
lags = [1, 5, 10, 20, 40, 60]
torch.manual_seed(0)
vanilla = nn.RNN(4, 32, batch_first=True)
g_rnn = grad_at_first_input(vanilla, lags)
# [3]
print("lag:      " + "   ".join(f"{l:>7d}" for l in lags))
print("|grad|:   " + "   ".join(f"{g:.1e}" for g in g_rnn))

plt.figure(figsize=(6.2, 3.2))
plt.semilogy(lags, g_rnn, "o-", color="#E57200", ms=5, label="vanilla RNN")
plt.xlabel("lag $T$ between output and first input")
plt.ylabel(r"$\|\partial L / \partial \mathbf{x}_1\|$")
plt.legend(); plt.tight_layout(); plt.show()

**Plan**

1. Prepare the inputs and fixed settings for the example.
2. Measure the LSTM gradient norm across lag.

In [ ]:
# [1]
torch.manual_seed(0)
lstm = nn.LSTM(4, 32, batch_first=True)
with torch.no_grad():
    n = lstm.bias_ih_l0.shape[0] // 4
    lstm.bias_ih_l0[n:2 * n].fill_(1.0)
    lstm.bias_hh_l0[n:2 * n].zero_()             # the two bias slices sum to +1
# [2]
g_lstm = grad_at_first_input(lstm, lags)

**Plan**

1. Define the reusable helpers: `make_recall`, `SeqClassifier`, and `train_recall`.
2. Prepare the inputs and fixed settings for the example.
3. Recall-the-first-token at lag 80, three configurations.

In [ ]:
# [1]
def make_recall(n: int, T: int, vocab: int = 4,
                seed: int = 0) -> tuple[torch.Tensor, torch.Tensor]:
    g = torch.Generator().manual_seed(seed)
    X = torch.randint(0, vocab, (n, T), generator=g)
    return F.one_hot(X, vocab).float(), X[:, 0]     # label = first token

class SeqClassifier(nn.Module):
    def __init__(self, cell: str, vocab=4, hidden=32, forget_bias=None):
        super().__init__()
        self.rec = {"rnn": nn.RNN, "lstm": nn.LSTM}[cell](vocab, hidden,
                                                          batch_first=True)
        if forget_bias is not None:
            with torch.no_grad():
                n = self.rec.bias_ih_l0.shape[0] // 4
                self.rec.bias_ih_l0[n:2 * n].fill_(forget_bias)
                self.rec.bias_hh_l0[n:2 * n].zero_()
        self.out = nn.Linear(hidden, vocab)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        o, _ = self.rec(x)
        return self.out(o[:, -1])                    # read out at the END only

def train_recall(cell: str, T: int, seed: int, forget_bias: float | None = None,
                 epochs: int = 80) -> float:
    torch.manual_seed(seed)
    net = SeqClassifier(cell, forget_bias=forget_bias)
    X_tr, y_tr = make_recall(2000, T, seed=1)
    X_te, y_te = make_recall(500, T, seed=2)
    opt = torch.optim.Adam(net.parameters(), lr=3e-3)
    for _ in range(epochs):
        perm = torch.randperm(len(X_tr))
        for i in range(0, len(X_tr), 128):
            idx = perm[i:i + 128]
            loss = F.cross_entropy(net(X_tr[idx]), y_tr[idx])
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            opt.step()
    with torch.no_grad():
        return (net(X_te).argmax(1) == y_te).float().mean().item(), net

# [2]
T = 80
configs = [("vanilla RNN", "rnn", None),
           ("LSTM, default init", "lstm", None),
           ("LSTM, forget bias +1", "lstm", 1.0)]
probe_nets: dict[str, nn.Module] = {}                # kept for the diagnostic below
print(f"recall accuracy at lag {T} (chance 25%), seeds 0 / 1 / 6050:")
# [3]
for label, cell, fb in configs:
    results = [train_recall(cell, T, s, fb) for s in [0, 1, 6050]]
    accs = [a for a, _ in results]
    probe_nets[label] = results[-1][1]                # seed 6050's model
    print(f"  {label:22s} " + "   ".join(f"{a:.0%}" for a in accs))

**Plan**

1. Define the reusable `forget_gate_trace` helper.
2. Prepare the inputs and fixed settings for the example.
3. Replay the trained LSTMs and read their forget gates.

In [ ]:
# [1]
@torch.no_grad()
def forget_gate_trace(net: SeqClassifier, T: int = 80) -> list[float]:
    X_probe, _ = make_recall(64, T, seed=3)           # fresh probe sequences
    lstm = net.rec
    W_ih, W_hh = lstm.weight_ih_l0, lstm.weight_hh_l0
    b = lstm.bias_ih_l0 + lstm.bias_hh_l0
    hidden = W_hh.shape[1]
    h = torch.zeros(64, hidden)
    c = torch.zeros(64, hidden)
    means = []
    for t in range(T):                                # replayed by hand
        z = X_probe[:, t] @ W_ih.T + h @ W_hh.T + b
        i_gate, f_gate, g_cand, o_gate = z.chunk(4, dim=1)
        f = torch.sigmoid(f_gate)
        c = f * c + torch.sigmoid(i_gate) * torch.tanh(g_cand)
        h = torch.sigmoid(o_gate) * torch.tanh(c)
        means.append(f.mean().item())
    out_ref, _ = lstm(X_probe)                        # replay must match the module
    assert torch.allclose(h, out_ref[:, -1], atol=1e-5)
    return means

# [2]
plt.figure(figsize=(6.2, 3.2))
# [3]
for label, color in [("LSTM, forget bias +1", "#232D4B"),
                     ("LSTM, default init", "#E57200")]:
    plt.plot(range(1, T + 1), forget_gate_trace(probe_nets[label]),
             color=color, lw=2, label=label)
plt.axhline(0.5, ls=":", color="#B8B8A8")
plt.ylim(0, 1.02)
plt.xlabel("timestep $t$")
plt.ylabel("mean forget gate")
plt.legend()
plt.tight_layout(); plt.show()

**Plan**

1. Prepare the inputs and fixed settings for the example.
2. Run a character-level LSTM trained on Chapters 1–9.
3. Report or visualize the measured result.
4. Define the `CharLSTM` module.

In [ ]:
import hashlib
from pathlib import Path

# [1]
corpus_path = Path("../../data/book-corpus-ch1-9.txt")
text = corpus_path.read_text(encoding="utf-8")
# [2]
assert len(text) == 148_594
assert hashlib.sha256(text.encode("utf-8")).hexdigest() == (
    "b0fc23a513e37e7bcd78da04a03877345f6ebc850b91dbae260656d9924fb299"
)
chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
data = torch.tensor([stoi[c] for c in text])
split = int(0.9 * len(data))
train_data, valid_data = data[:split], data[split:]
# [3]
print(f"corpus: 9 chapters, {len(text):,} characters, vocab {len(chars)}")
print(f"deterministic split: {len(train_data):,} train / {len(valid_data):,} held out")

# [4]
class CharLSTM(nn.Module):
    def __init__(self, vocab: int, hidden: int = 128):
        super().__init__()
        self.vocab = vocab
        self.lstm = nn.LSTM(vocab, hidden, batch_first=True)
        self.out = nn.Linear(hidden, vocab)
    def forward(
        self, x: torch.Tensor,
        state: tuple[torch.Tensor, torch.Tensor] | None = None,
    ) -> tuple[torch.Tensor, tuple[torch.Tensor, torch.Tensor]]:
        o, state = self.lstm(F.one_hot(x, self.vocab).float(), state)
        return self.out(o), state                      # logits at EVERY step

**Plan**

1. Declare the shared next-token training interface and its protocol controls.
2. Configure the optimizer and requested step budget.
3. Draw or reuse chunk starts, then construct shifted input–target pairs.
4. Predict every next token and average cross-entropy across batch and time.
5. Backpropagate, clip the gradient, and take one optimizer step.
6. Retain a sparse learning curve and return the trained artifact.

In [ ]:
"""Next-token training loop — Chapter 10's listing, importable.

The loop is printed and taught in Chapter 10 (truncated-BPTT chunk sampling,
gradient clipping); later chapters import it and print only their deltas.
"""
import torch
import torch.nn.functional as F
from torch import nn


# [1]
def fit_next_token(
    model: nn.Module,
    data: torch.Tensor,
    *,
    vocab: int,
    context: int = 100,
    batch: int = 64,
    steps: int = 2501,
    lr: float = 2e-3,
    clip: float = 1.0,
    schedule: list[torch.Tensor] | None = None,
    log_every: int = 500,
    log_decimals: int = 2,
) -> tuple[nn.Module, list[tuple[int, float]]]:
    """Train `model` to predict data[t+1] from data[t-context+1 .. t].

    With `schedule=None`, chunk starts are drawn fresh each step (Chapter 10's
    truncated-BPTT sampling, consuming the global RNG exactly as printed there).
    A precomputed `schedule` of start tensors makes the minibatch order an
    explicit, shareable part of the protocol (Chapter 14's paired comparison).
    """
    # [2]
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    curve: list[tuple[int, float]] = []
    n_steps = len(schedule) if schedule is not None else steps
    # [3]
    for step in range(n_steps):
        starts = (
            schedule[step]
            if schedule is not None
            else torch.randint(0, len(data) - context - 1, (batch,))
        )
        xb = torch.stack([data[j : j + context] for j in starts])
        yb = torch.stack([data[j + 1 : j + context + 1] for j in starts])
        # [4]
        logits = model(xb)
        if isinstance(logits, tuple):          # recurrent models return state
            logits = logits[0]
        loss = F.cross_entropy(logits.reshape(-1, vocab), yb.reshape(-1))
        # [5]
        opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), clip)
        opt.step()
        # [6]
        if step % log_every == 0:
            curve.append((step, loss.item()))
            print(f"step {step:4d}   loss {loss.item():.{log_decimals}f}")
    return model, curve

**Plan**

1. Declare inference-only fixed-window evaluation.
2. Switch to evaluation mode and initialize token-weighted totals.
3. Traverse consecutive non-overlapping windows.
4. Predict with recurrent state reset at each window boundary.
5. Sum loss within windows and count all scored tokens.
6. Divide once to obtain mean next-token cross-entropy.

In [ ]:
"""Fixed-window evaluation loss — Chapter 10's protocol, importable.

Scores a sequence in consecutive `window`-sized chunks with the recurrent (or
attention) state reset at each boundary, so train and held-out numbers are
comparable across chapters and architectures.
"""
import torch
import torch.nn.functional as F
from torch import nn


# [1]
@torch.no_grad()
def fixed_window_loss(
    net: nn.Module,
    sequence: torch.Tensor,
    *,
    vocab: int,
    window: int = 100,
) -> float:
    """Mean next-token cross-entropy over non-overlapping windows."""
    # [2]
    net.eval()
    total_loss, n_tokens = 0.0, 0
    # [3]
    for start in range(0, len(sequence) - 1, window):
        stop = min(start + window, len(sequence) - 1)
        xb = sequence[start:stop].unsqueeze(0)
        yb = sequence[start + 1 : stop + 1]
        # [4]
        logits = net(xb)
        if isinstance(logits, tuple):      # recurrent models return state
            logits = logits[0]             # state resets at each window
        # [5]
        total_loss += F.cross_entropy(
            logits.reshape(-1, vocab), yb, reduction="sum"
        ).item()
        n_tokens += yb.numel()
    # [6]
    return total_loss / n_tokens

**Plan**

1. Prepare the inputs and fixed settings for the example.
2. Run char lm train run.
3. Report or visualize the measured result.

In [ ]:
from dlbook.training import fit_next_token   # Listing 10.1, imported

# [1]
torch.manual_seed(6050)
model = CharLSTM(len(chars))
# [2]
model, _ = fit_next_token(model, train_data, vocab=len(chars))

from dlbook.evaluation import fixed_window_loss   # Listing 10.2, imported

# [3]
held_out_loss = fixed_window_loss(model, valid_data, vocab=len(chars))
print(f"held-out fixed-window loss {held_out_loss:.2f}")

**Plan**

1. Define the reusable `sample` helper.
2. Sample from the model, one character at a time.

In [ ]:
# [1]
@torch.no_grad()
def sample(prompt: str, n: int = 300, temp: float = 0.8) -> str:
    model.eval()
    idx = torch.tensor([[stoi.get(c, 0) for c in prompt]])
    logits, state = model(idx)
    out = prompt
    for _ in range(n):
        p = F.softmax(logits[0, -1] / temp, -1)
        nxt = torch.multinomial(p, 1)[None]
        out += chars[nxt.item()]
        logits, state = model(nxt, state)              # consume each sample once
    return out

# [2]
print(sample("The gradient ", n=300))

**Plan**

1. Load the nine pinned WikiText-2 Rivanna records.
2. Summarize parameter count and held-out perplexity by model size.

In [ ]:
# [1]
import json
from pathlib import Path

wikitext_root = Path("../../experiments/rivanna/results/wikitext")
wikitext_records = [
    json.loads(path.read_text()) for path in sorted(wikitext_root.glob("*.json"))
]
wikitext_order = ["small", "medium", "large"]

# [2]
wikitext_summary = {}
for size in wikitext_order:
    rows = [row for row in wikitext_records if row["size"] == size]
    perplexities = torch.tensor([row["test_perplexity"] for row in rows])
    wikitext_summary[size] = {
        "parameters": rows[0]["parameter_count"],
        "values": perplexities,
        "mean": perplexities.mean().item(),
        "sd": perplexities.std(unbiased=True).item(),
    }